[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/ai-foundations/blob/main/07_images_as_data/tutorial.ipynb)

## ▶️ Run this notebook in Google Colab

**Option A — One click (recommended):** Click the **Open in Colab** badge above. It opens this notebook directly in a free cloud runtime — nothing to install locally.

**Option B — Upload manually:**
1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this `tutorial.ipynb`.

**Then, in Colab:**
1. Run the **first code cell** (the dependency-setup cell) — it installs everything this notebook needs (including OpenCV) directly into the Colab kernel. No `pip install`, `uv sync`, or `pyproject.toml` required.
2. Run the remaining cells top to bottom via **Runtime → Run all** (or `Ctrl/Cmd+F9`).

> 💡 This notebook is **fully self-contained**: the sample images it uses are generated inside the notebook, so it runs end-to-end in Colab with no external files.

---

In [ ]:
# === Inline dependency setup (self-contained; no `uv sync` / pyproject.toml needed) ===
# Installs this notebook's libraries directly into the running kernel.
# Works in local Jupyter, VS Code, and Google Colab. Safe to re-run (idempotent).
import sys, subprocess

_DEPS = [
    'python-dotenv>=1.0.0',
    'numpy>=1.26.0',
    'opencv-python>=4.8.0',
    'matplotlib>=3.7',
]

# Ensure pip is available in this kernel (some minimal venvs ship without it), then install.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    import ensurepip; ensurepip.bootstrap()

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_DEPS])
print('\u2713 Dependencies installed:', ', '.join(_DEPS))


In [ ]:
# === Google Colab: native Mermaid diagram rendering ===
# Colab does not render ```mermaid Markdown blocks. This helper injects the
# official Mermaid.js library into a cell's output and draws the diagram there.
# The Markdown cells keep their ```mermaid blocks so GitHub still renders them;
# the code cells below each diagram call render_mermaid(...) so Colab does too.
# Safe to re-run.
import IPython.display as _ipd

_MERMAID_COUNTER = 0


def render_mermaid(graph: str):
    # Render a Mermaid diagram in the current cell's output (works in Colab).
    global _MERMAID_COUNTER
    _MERMAID_COUNTER += 1
    container_id = f"mermaid-diagram-{_MERMAID_COUNTER}"
    # The graph text is placed verbatim inside the .mermaid div; Mermaid reads it
    # as textContent, so it must NOT be HTML-escaped.
    html = f'''
<div id="{container_id}" class="mermaid">
{graph}
</div>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({{ startOnLoad: false }});
  const el = document.getElementById("{container_id}");
  await mermaid.run({{ nodes: [el] }});
</script>
'''
    _ipd.display(_ipd.HTML(html))


# Module 7: Images as Data (OpenCV)

## AI Research Foundations Minicourse

**Learning Objectives:**
- Understand images as 3D NumPy arrays (Height × Width × Channels)
- Load and manipulate images using OpenCV
- Convert between BGR and RGB color spaces
- Resize images to standard dimensions for neural networks
- Normalize pixel values for optimal model training
- Process batches of images efficiently

---

## Setup

First, let's import the required libraries and configure our environment.

In [ ]:
# Environment setup
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# Core libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Configure matplotlib for inline display
%matplotlib inline

# Set larger figure sizes for better visibility
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ Setup complete")
print(f"NumPy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")

## 1. Introduction: Images as Numbers

To a computer, an image is not a visual object — it's a **3D array of numbers**.

### Image Array Structure

```mermaid
graph TD
    A[Image] --> B[3D Array]
    B --> C[Dimension 1: Height rows]
    B --> D[Dimension 2: Width columns]
    B --> E[Dimension 3: Color Channels]
    C --> F[480 pixels tall]
    D --> G[640 pixels wide]
    E --> H[3 channels: R, G, B]
    H --> I[Shape: 480, 640, 3]
```

> **Key Insight**: Every image is a 3D array with shape `(Height, Width, Channels)`. Each pixel contains 3 numbers (0-255) representing Red, Green, and Blue intensity.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph TD
    A[Image] --> B[3D Array]
    B --> C[Dimension 1: Height rows]
    B --> D[Dimension 2: Width columns]
    B --> E[Dimension 3: Color Channels]
    C --> F[480 pixels tall]
    D --> G[640 pixels wide]
    E --> H[3 channels: R, G, B]
    H --> I[Shape: 480, 640, 3]
''')


## 2. What is an Image to a Computer?

### Pixels as Numbers

Each pixel in a color image contains three values:
- **0** = no intensity (black)
- **255** = maximum intensity (brightest)
- Values between 0-255 create all possible colors

### Grayscale vs Color

- **Grayscale**: 2D array `(H, W)` — single intensity value per pixel
- **Color**: 3D array `(H, W, 3)` — three values per pixel (R, G, B)

Let's create synthetic images to demonstrate these concepts.

In [ ]:
# Create a simple synthetic grayscale image
# Each pixel has a single value from 0 (black) to 255 (white)

grayscale_img = np.zeros((100, 200), dtype=np.uint8)

# Create a horizontal gradient from black to white
for col in range(200):
    grayscale_img[:, col] = int((col / 200) * 255)

print("Grayscale Image:")
print(f"Shape: {grayscale_img.shape}")
print(f"Data type: {grayscale_img.dtype}")
print(f"Value range: {grayscale_img.min()} to {grayscale_img.max()}")
print(f"\nFirst row values (first 10 pixels): {grayscale_img[0, :10]}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.imshow(grayscale_img, cmap='gray', vmin=0, vmax=255)
ax.set_title('Grayscale Image: 2D Array (H × W)\nBlack (0) to White (255)', fontsize=12)
ax.set_xlabel('Width (200 pixels)')
ax.set_ylabel('Height (100 pixels)')
ax.grid(False)
plt.colorbar(ax.images[0], ax=ax, label='Pixel Intensity')
plt.tight_layout()
plt.show()

In [ ]:
# Create a synthetic color (RGB) image
# Each pixel has THREE values: [Red, Green, Blue]

color_img = np.zeros((100, 200, 3), dtype=np.uint8)

# Create separate gradients for each channel
for row in range(100):
    for col in range(200):
        color_img[row, col, 0] = int((row / 100) * 255)      # Red: vertical gradient
        color_img[row, col, 1] = int((col / 200) * 255)      # Green: horizontal gradient
        color_img[row, col, 2] = 128                          # Blue: constant

print("Color Image (RGB):")
print(f"Shape: {color_img.shape}")
print(f"Data type: {color_img.dtype}")
print(f"Dimensions: Height={color_img.shape[0]}, Width={color_img.shape[1]}, Channels={color_img.shape[2]}")

# Sample a pixel from the center
center_pixel = color_img[50, 100]
print(f"\nCenter pixel [50, 100]:")
print(f"  Red:   {center_pixel[0]}")
print(f"  Green: {center_pixel[1]}")
print(f"  Blue:  {center_pixel[2]}")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Full RGB image
axes[0].imshow(color_img)
axes[0].set_title('Full Color Image\n(RGB)', fontsize=11)
axes[0].set_xlabel('Width')
axes[0].set_ylabel('Height')
axes[0].grid(False)

# Individual channels
channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
channel_colors = ['Reds', 'Greens', 'Blues']

for i in range(3):
    axes[i+1].imshow(color_img[:, :, i], cmap=channel_colors[i], vmin=0, vmax=255)
    axes[i+1].set_title(f'{channel_names[i]}\n(Layer {i})', fontsize=11)
    axes[i+1].set_xlabel('Width')
    if i == 0:
        axes[i+1].set_ylabel('Height')
    axes[i+1].grid(False)

plt.tight_layout()
plt.show()

> **Key Insight**: A color image is essentially three grayscale images stacked together — one for Red, one for Green, and one for Blue. The combination of these three channels creates all visible colors.

## 3. Loading Images with OpenCV

OpenCV (`cv2`) is the most popular computer vision library. Let's use it to load real images.

First, we need to generate some sample images to work with.

In [ ]:
# Inline sample-image generator (self-contained; no external .py file needed)
# Uses only numpy, cv2, and os -- all imported above.
import os
import numpy as np
import cv2


def create_sample_images(output_dir: str = "data/images", num_images: int = 8):
    """Generate synthetic sample images with different sizes and patterns."""

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    print(f"Generating {num_images} synthetic sample images in {output_dir}/")
    print("=" * 60)

    # Image 1: Red gradient (320x240)
    img1 = np.zeros((240, 320, 3), dtype=np.uint8)
    for i in range(240):
        img1[i, :, 2] = int((i / 240) * 255)  # Red channel gradient
    cv2.imwrite(os.path.join(output_dir, "sample_01_red_gradient.jpg"), img1)
    print("✓ sample_01_red_gradient.jpg (320x240) - vertical red gradient")

    # Image 2: Green and Blue diagonal (480x360)
    img2 = np.zeros((360, 480, 3), dtype=np.uint8)
    for i in range(360):
        for j in range(480):
            img2[i, j, 0] = int(((i + j) / (360 + 480)) * 255)  # Blue
            img2[i, j, 1] = int(((i + j) / (360 + 480)) * 128)  # Green
    cv2.imwrite(os.path.join(output_dir, "sample_02_blue_green_diagonal.jpg"), img2)
    print("✓ sample_02_blue_green_diagonal.jpg (480x360) - diagonal gradient")

    # Image 3: Random noise (256x256)
    img3 = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
    cv2.imwrite(os.path.join(output_dir, "sample_03_random_noise.jpg"), img3)
    print("✓ sample_03_random_noise.jpg (256x256) - random RGB noise")

    # Image 4: Colored rectangles (640x480)
    img4 = np.zeros((480, 640, 3), dtype=np.uint8)
    # Red rectangle
    img4[50:200, 50:250, 2] = 255
    # Green rectangle
    img4[280:430, 150:350, 1] = 255
    # Blue rectangle
    img4[150:300, 390:590, 0] = 255
    cv2.imwrite(os.path.join(output_dir, "sample_04_rectangles.jpg"), img4)
    print("✓ sample_04_rectangles.jpg (640x480) - three colored rectangles")

    # Image 5: Checkerboard pattern (512x512)
    img5 = np.zeros((512, 512, 3), dtype=np.uint8)
    square_size = 64
    for i in range(0, 512, square_size):
        for j in range(0, 512, square_size):
            if ((i // square_size) + (j // square_size)) % 2 == 0:
                img5[i:i+square_size, j:j+square_size] = [255, 255, 255]
    cv2.imwrite(os.path.join(output_dir, "sample_05_checkerboard.jpg"), img5)
    print("✓ sample_05_checkerboard.jpg (512x512) - black and white checkerboard")

    # Image 6: Radial gradient (300x300)
    img6 = np.zeros((300, 300, 3), dtype=np.uint8)
    center_x, center_y = 150, 150
    max_dist = np.sqrt(center_x**2 + center_y**2)
    for i in range(300):
        for j in range(300):
            dist = np.sqrt((i - center_y)**2 + (j - center_x)**2)
            value = int((1 - dist / max_dist) * 255)
            img6[i, j] = [value, value // 2, value]  # Purple gradient
    cv2.imwrite(os.path.join(output_dir, "sample_06_radial_gradient.jpg"), img6)
    print("✓ sample_06_radial_gradient.jpg (300x300) - radial purple gradient")

    # Image 7: Horizontal stripes (400x300)
    img7 = np.zeros((300, 400, 3), dtype=np.uint8)
    stripe_height = 50
    colors = [
        [255, 0, 0],    # Blue
        [0, 255, 0],    # Green
        [0, 0, 255],    # Red
        [255, 255, 0],  # Cyan
        [255, 0, 255],  # Magenta
        [0, 255, 255],  # Yellow
    ]
    for i, color in enumerate(colors):
        img7[i*stripe_height:(i+1)*stripe_height, :] = color
    cv2.imwrite(os.path.join(output_dir, "sample_07_stripes.jpg"), img7)
    print("✓ sample_07_stripes.jpg (400x300) - horizontal color stripes")

    # Image 8: Random colored circles (600x400)
    img8 = np.ones((400, 600, 3), dtype=np.uint8) * 240  # Light gray background
    np.random.seed(42)
    for _ in range(10):
        center = (np.random.randint(50, 550), np.random.randint(50, 350))
        radius = np.random.randint(20, 60)
        color = tuple(np.random.randint(0, 256, 3).tolist())
        cv2.circle(img8, center, radius, color, -1)
    cv2.imwrite(os.path.join(output_dir, "sample_08_circles.jpg"), img8)
    print("✓ sample_08_circles.jpg (600x400) - random colored circles")

    print("=" * 60)
    print(f"✓ All {num_images} sample images generated successfully!\n")

In [ ]:
# Generate sample images if they don't exist
# (create_sample_images was defined inline in the cell above -- no external file needed)
images_dir = Path("data/images")
images_dir.mkdir(parents=True, exist_ok=True)

if not any(images_dir.glob('*.jpg')):
    print("Generating sample images...")
    create_sample_images()
else:
    print(f"✓ Sample images found in {images_dir}/")

# List available images
image_files = sorted(images_dir.glob('*.jpg'))
print(f"\nAvailable images ({len(image_files)}):")
for img_file in image_files:
    print(f"  - {img_file.name}")

In [ ]:
# Load the first image
image_path = str(image_files[0])
img = cv2.imread(image_path)

print(f"Loading: {image_files[0].name}")
print(f"\n✓ Image loaded successfully")
print(f"\nImage properties:")
print(f"  Shape:      {img.shape}")
print(f"  Data type:  {img.dtype}")
print(f"  Height:     {img.shape[0]} pixels")
print(f"  Width:      {img.shape[1]} pixels")
print(f"  Channels:   {img.shape[2]}")
print(f"\nValue statistics:")
print(f"  Min:        {img.min()}")
print(f"  Max:        {img.max()}")
print(f"  Mean:       {img.mean():.2f}")
print(f"  Memory:     {img.nbytes:,} bytes ({img.nbytes / 1024:.2f} KB)")

### Understanding the Shape: (H, W, C)

The shape tuple `(H, W, C)` tells us everything about the image structure:

```
img.shape = (240, 320, 3)
             │    │    └─ 3 channels (BGR)
             │    └────── 320 pixels wide (columns)
             └─────────── 240 pixels tall (rows)
```

Total pixels = 240 × 320 = 76,800 pixels  
Total values = 76,800 × 3 = 230,400 numbers stored in memory

## 4. The BGR vs RGB Problem

**⚠️ IMPORTANT**: OpenCV loads images in **BGR** (Blue, Green, Red) format, NOT RGB!

This is a historical quirk from early video capture cards. Most other libraries (matplotlib, PIL, PyTorch, TensorFlow) expect **RGB** (Red, Green, Blue) format.

### The Problem

If you display an OpenCV image directly with matplotlib, the colors will be wrong:
- Red objects appear blue
- Blue objects appear red
- Green stays the same

### The Solution

Always convert BGR → RGB using `cv2.cvtColor()`

In [ ]:
# Load an image that clearly shows the BGR problem
# Let's use the red gradient image
red_gradient_path = str(images_dir / "sample_01_red_gradient.jpg")
img_bgr = cv2.imread(red_gradient_path)

print(f"Loaded: {Path(red_gradient_path).name}")
print(f"This image should be RED (gradually getting brighter from top to bottom)\n")

# Sample a pixel to show BGR ordering
sample_pixel_bgr = img_bgr[200, 160]  # Near bottom center (should be bright red)
print(f"Sample pixel at [200, 160] in BGR format:")
print(f"  B (Blue):  {sample_pixel_bgr[0]}")
print(f"  G (Green): {sample_pixel_bgr[1]}")
print(f"  R (Red):   {sample_pixel_bgr[2]}")
print(f"\nNotice: The RED channel (index 2) has the high value!")

In [ ]:
# Display BGR image (WRONG COLORS)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Display as-is (BGR) - colors will be wrong!
axes[0].imshow(img_bgr)
axes[0].set_title('❌ WRONG: BGR displayed as RGB\n(Red gradient appears BLUE!)', 
                   fontsize=12, color='red', fontweight='bold')
axes[0].set_xlabel('Width')
axes[0].set_ylabel('Height')
axes[0].grid(False)

# Convert BGR to RGB
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Display RGB (CORRECT)
axes[1].imshow(img_rgb)
axes[1].set_title('✓ CORRECT: BGR → RGB conversion\n(Red gradient appears RED!)', 
                   fontsize=12, color='green', fontweight='bold')
axes[1].set_xlabel('Width')
axes[1].set_ylabel('Height')
axes[1].grid(False)

plt.tight_layout()
plt.show()

# Show the pixel values after conversion
sample_pixel_rgb = img_rgb[200, 160]
print(f"\nAfter BGR → RGB conversion:")
print(f"  R (Red):   {sample_pixel_rgb[0]}  ← Now in position 0!")
print(f"  G (Green): {sample_pixel_rgb[1]}")
print(f"  B (Blue):  {sample_pixel_rgb[2]}  ← Now in position 2!")
print(f"\nOnly the ORDER changed, not the actual values.")

> **Key Insight**: Always use `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` immediately after loading an image with OpenCV. This swaps the Red and Blue channels to match what most libraries expect.

## 5. Image Resizing

### Why Resize?

Neural networks require **fixed-size inputs**. All images in a batch must have the same dimensions.

Real-world images come in many different sizes:
- Phone photos: 3024×4032
- Web images: 800×600, 1920×1080
- Thumbnails: 128×128

### Standard Sizes

The most common size is **224×224** pixels:
- Used by ImageNet dataset
- Expected by VGG, ResNet, MobileNet, and many other pretrained models
- Good balance between detail and computation

### The Resizing Process

```mermaid
graph LR
    A[Original Image<br/>480 × 640 × 3] --> B[cv2.resize]
    B --> C[Resized Image<br/>224 × 224 × 3]
    B -.-> D[Interpolation:<br/>INTER_LINEAR,<br/>INTER_AREA,<br/>INTER_CUBIC]
    C --> E[Fixed size for<br/>neural network]
```

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A[Original Image<br/>480 × 640 × 3] --> B[cv2.resize]
    B --> C[Resized Image<br/>224 × 224 × 3]
    B -.-> D[Interpolation:<br/>INTER_LINEAR,<br/>INTER_AREA,<br/>INTER_CUBIC]
    C --> E[Fixed size for<br/>neural network]
''')


In [ ]:
# Load an image with non-standard size
large_img_path = str(images_dir / "sample_04_rectangles.jpg")
img_original = cv2.imread(large_img_path)
img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)  # Fix BGR

print(f"Original image: {Path(large_img_path).name}")
print(f"  Shape: {img_original.shape}")
print(f"  Size: {img_original.shape[1]}×{img_original.shape[0]} (W×H)")
print(f"  Pixels: {img_original.shape[0] * img_original.shape[1]:,}")

# Resize to standard 224×224
target_size = (224, 224)  # (width, height) - note the order!
img_resized = cv2.resize(img_original, target_size, interpolation=cv2.INTER_LINEAR)

print(f"\nAfter resizing to {target_size[0]}×{target_size[1]}:")
print(f"  Shape: {img_resized.shape}")
print(f"  Size: {img_resized.shape[1]}×{img_resized.shape[0]} (W×H)")
print(f"  Pixels: {img_resized.shape[0] * img_resized.shape[1]:,}")

# Calculate size change
original_pixels = img_original.shape[0] * img_original.shape[1]
resized_pixels = img_resized.shape[0] * img_resized.shape[1]
reduction_ratio = resized_pixels / original_pixels

print(f"\nSize reduction: {reduction_ratio:.2%} of original")
print(f"Memory saved: {(1 - reduction_ratio) * 100:.1f}%")

In [ ]:
# Visualize original vs resized
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_original)
axes[0].set_title(f'Original Image\n{img_original.shape[1]}×{img_original.shape[0]} pixels\n'
                   f'{img_original.shape[0] * img_original.shape[1]:,} total pixels', fontsize=11)
axes[0].set_xlabel('Width')
axes[0].set_ylabel('Height')
axes[0].grid(False)

axes[1].imshow(img_resized)
axes[1].set_title(f'Resized Image\n{img_resized.shape[1]}×{img_resized.shape[0]} pixels\n'
                   f'{img_resized.shape[0] * img_resized.shape[1]:,} total pixels', fontsize=11)
axes[1].set_xlabel('Width')
axes[1].set_ylabel('Height')
axes[1].grid(False)

plt.tight_layout()
plt.show()

### Interpolation Methods

When resizing, OpenCV must create new pixel values. Different interpolation methods have different trade-offs:

| Method | Best For | Speed | Quality |
|--------|----------|-------|----------|
| `INTER_NEAREST` | Pixel art, fast preview | Fastest | Blocky |
| `INTER_LINEAR` | General resizing | Fast | Good |
| `INTER_AREA` | Shrinking images | Medium | Best for shrinking |
| `INTER_CUBIC` | High-quality enlargement | Slow | Smooth |
| `INTER_LANCZOS4` | Professional work | Slowest | Highest quality |

For machine learning preprocessing: **INTER_LINEAR** (default) or **INTER_AREA** (when shrinking)

> **Key Insight**: Resizing is essential for creating uniform inputs for neural networks. The standard 224×224 size is a historical convention from ImageNet, but modern architectures can handle other sizes (e.g., 299×299 for Inception, 512×512 for high-resolution tasks).

## 6. Pixel Normalization

### Why Normalize?

Raw pixel values (0-255) are not ideal for neural networks:

1. **Large values** → Large gradients → Unstable training
2. **Inconsistent scale** → Some features dominate learning
3. **Poor convergence** → Training takes longer or fails

### The Solution

Normalize pixels to a smaller range, typically **0.0 to 1.0**:

```python
normalized = img.astype(np.float32) / 255.0
```

This transformation:
- 0 (black) → 0.0
- 128 (mid-gray) → 0.502 (≈0.5)
- 255 (white) → 1.0

### Normalization Process

```mermaid
graph LR
    A[Raw Pixels<br/>uint8<br/>0-255] --> B[Convert to<br/>float32]
    B --> C[Divide by<br/>255.0]
    C --> D[Normalized Pixels<br/>float32<br/>0.0-1.0]
    D --> E[Ready for<br/>neural network]
```

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A[Raw Pixels<br/>uint8<br/>0-255] --> B[Convert to<br/>float32]
    B --> C[Divide by<br/>255.0]
    C --> D[Normalized Pixels<br/>float32<br/>0.0-1.0]
    D --> E[Ready for<br/>neural network]
''')


In [ ]:
# Start with our resized image
print("Before normalization:")
print(f"  Data type: {img_resized.dtype}")
print(f"  Value range: {img_resized.min()} to {img_resized.max()}")
print(f"  Mean value: {img_resized.mean():.2f}")
print(f"  Memory: {img_resized.nbytes:,} bytes")

# Normalize: convert to float32 and divide by 255.0
img_normalized = img_resized.astype(np.float32) / 255.0

print("\nAfter normalization:")
print(f"  Data type: {img_normalized.dtype}")
print(f"  Value range: {img_normalized.min():.3f} to {img_normalized.max():.3f}")
print(f"  Mean value: {img_normalized.mean():.3f}")
print(f"  Memory: {img_normalized.nbytes:,} bytes (4x larger due to float32)")

# Sample pixels before and after
sample_row, sample_col = 100, 100
pixel_before = img_resized[sample_row, sample_col]
pixel_after = img_normalized[sample_row, sample_col]

print(f"\nSample pixel at [{sample_row}, {sample_col}]:")
print(f"  Before (uint8):  [{pixel_before[0]:3d}, {pixel_before[1]:3d}, {pixel_before[2]:3d}]")
print(f"  After (float32): [{pixel_after[0]:.3f}, {pixel_after[1]:.3f}, {pixel_after[2]:.3f}]")

In [ ]:
# Visualize pixel value distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of raw pixel values
axes[0].hist(img_resized.flatten(), bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[0].set_title('Raw Pixel Values\n(uint8: 0-255)', fontsize=12)
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 255)
axes[0].axvline(img_resized.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {img_resized.mean():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram of normalized pixel values
axes[1].hist(img_normalized.flatten(), bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title('Normalized Pixel Values\n(float32: 0.0-1.0)', fontsize=12)
axes[1].set_xlabel('Pixel Value')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 1)
axes[1].axvline(img_normalized.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {img_normalized.mean():.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: The distribution shape stays the same, only the scale changes.")

### Benefits of Normalization

1. **Faster Convergence**: Gradient descent optimizes better with smaller values
2. **Numerical Stability**: Prevents overflow/underflow in calculations
3. **Consistent Scale**: All features in the same range
4. **Model Compatibility**: Required by most pretrained models
5. **Better Regularization**: Weights stay in a reasonable range

> **Key Insight**: Always normalize pixel values after resizing and before feeding into a neural network. The standard normalization is division by 255.0, but some models also subtract the ImageNet mean and divide by ImageNet standard deviation for each channel.

## 7. Processing a Batch of Images

In practice, we rarely process just one image. Neural networks train on **batches** of images for efficiency.

### The Complete Pipeline

```mermaid
graph TD
    A[Folder of Images<br/>Various sizes] --> B[For each image:]
    B --> C[1. Load with cv2.imread]
    C --> D[2. Convert BGR → RGB]
    D --> E[3. Resize to 224×224]
    E --> F[4. Normalize to 0-1]
    F --> G[5. Append to list]
    G --> H{More images?}
    H -->|Yes| B
    H -->|No| I[Stack into batch array<br/>np.stack axis=0]
    I --> J[Batch array<br/>N, 224, 224, 3]
    J --> K[Ready for model training]
```

Let's process all our sample images into a single batch.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph TD
    A[Folder of Images<br/>Various sizes] --> B[For each image:]
    B --> C[1. Load with cv2.imread]
    C --> D[2. Convert BGR → RGB]
    D --> E[3. Resize to 224×224]
    E --> F[4. Normalize to 0-1]
    F --> G[5. Append to list]
    G --> H{More images?}
    H -->|Yes| B
    H -->|No| I[Stack into batch array<br/>np.stack axis=0]
    I --> J[Batch array<br/>N, 224, 224, 3]
    J --> K[Ready for model training]
''')


In [ ]:
def preprocess_image(image_path: Path, target_size=(224, 224)):
    """
    Complete preprocessing pipeline for a single image.
    
    Steps:
    1. Load image (BGR format)
    2. Convert BGR → RGB
    3. Resize to target size
    4. Normalize to 0-1 range
    
    Returns:
        Preprocessed image as float32 array with shape (H, W, C) and values in [0, 1]
    """
    # 1. Load
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")
    
    # 2. BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 3. Resize
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    
    # 4. Normalize
    img = img.astype(np.float32) / 255.0
    
    return img


# Process all images
target_size = (224, 224)
print(f"Processing {len(image_files)} images to {target_size[0]}×{target_size[1]}...\n")

processed_images = []

for i, img_path in enumerate(image_files, 1):
    # Load original to show size
    img_original = cv2.imread(str(img_path))
    original_shape = img_original.shape
    
    # Preprocess
    img_processed = preprocess_image(img_path, target_size)
    processed_images.append(img_processed)
    
    print(f"{i:2d}. {img_path.name:30s} {original_shape[1]:4d}×{original_shape[0]:4d} → "
          f"{img_processed.shape[1]}×{img_processed.shape[0]}")

print(f"\n✓ All {len(processed_images)} images preprocessed successfully")

In [ ]:
# Stack into a batch array
batch = np.stack(processed_images, axis=0)

print("Batch Array:")
print(f"  Shape: {batch.shape}")
print(f"\nDimensions breakdown:")
print(f"  Axis 0 (N): {batch.shape[0]} images in the batch")
print(f"  Axis 1 (H): {batch.shape[1]} pixels tall")
print(f"  Axis 2 (W): {batch.shape[2]} pixels wide")
print(f"  Axis 3 (C): {batch.shape[3]} color channels (R, G, B)")

print(f"\nData properties:")
print(f"  Data type:  {batch.dtype}")
print(f"  Value range: {batch.min():.3f} to {batch.max():.3f}")
print(f"  Mean:       {batch.mean():.3f}")
print(f"  Std dev:    {batch.std():.3f}")

print(f"\n🎯 This is exactly what a neural network expects as input!")

In [ ]:
# Visualize the batch
n_images = len(processed_images)
n_cols = 4
n_rows = (n_images + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten() if n_images > 1 else [axes]

for i in range(n_images):
    axes[i].imshow(batch[i])
    axes[i].set_title(f'Image {i+1}\nShape: {batch[i].shape}', fontsize=10)
    axes[i].axis('off')

# Hide extra subplots
for i in range(n_images, len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Preprocessed Batch: {batch.shape[0]} images × {batch.shape[1]}×{batch.shape[2]}×{batch.shape[3]}',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

> **Key Insight**: `np.stack()` combines individual images into a single 4D array. The first dimension (axis 0) indexes which image in the batch. This is the standard format for feeding data into neural networks.

## 8. Memory Considerations

Image data can consume significant memory, especially when training deep learning models.

### Memory Calculation

For a batch of images:

```
Memory (bytes) = N × H × W × C × bytes_per_value
```

Where:
- N = number of images in batch
- H = height in pixels
- W = width in pixels
- C = number of channels (3 for RGB)
- bytes_per_value = 4 (for float32) or 1 (for uint8)

### Example

For our current batch:
- 8 images × 224 × 224 × 3 × 4 bytes = **1,204,224 bytes ≈ 1.15 MB**

This seems small, but consider typical training scenarios:

In [ ]:
# Calculate memory for our current batch
N, H, W, C = batch.shape
total_bytes = batch.nbytes
total_mb = total_bytes / (1024 * 1024)

print(f"Current batch memory:")
print(f"  Shape:       {batch.shape}")
print(f"  Total bytes: {total_bytes:,} bytes")
print(f"  Total MB:    {total_mb:.2f} MB")
print(f"  Per image:   {total_bytes / N:,.0f} bytes ({total_mb / N:.3f} MB)")

# Calculate for typical batch sizes
print(f"\nMemory for different batch sizes (224×224×3, float32):")
print(f"\n{'Batch Size':>12} | {'Memory (MB)':>12} | {'Memory (GB)':>12}")
print("-" * 42)

batch_sizes = [8, 16, 32, 64, 128, 256, 512, 1024]
bytes_per_image = H * W * C * 4  # float32 = 4 bytes

for bs in batch_sizes:
    batch_mb = (bs * bytes_per_image) / (1024 * 1024)
    batch_gb = batch_mb / 1024
    print(f"{bs:>12,} | {batch_mb:>12.2f} | {batch_gb:>12.3f}")

print(f"\n⚠️  GPU Memory Limits:")
print(f"  Consumer GPU:  4-8 GB (RTX 3060, 3070)")
print(f"  High-end GPU:  12-24 GB (RTX 3090, 4090, A6000)")
print(f"  Data center:   40-80 GB (A100, H100)")
print(f"\n💡 Batch size must fit in GPU memory along with model weights and gradients!")

### Why Memory Matters

1. **GPU Memory is Limited**: Unlike RAM, GPU memory is precious
2. **Model Weights Take Space**: A ResNet-50 uses ~100 MB
3. **Gradients Double It**: Backpropagation stores gradients (~2× model size)
4. **Intermediate Activations**: Feature maps from each layer need memory

**Rule of Thumb**: Your batch should use <50% of available GPU memory to leave room for model and gradients.

### Memory Optimization Strategies

1. **Reduce Batch Size**: Most common solution
2. **Smaller Images**: 224×224 → 96×96 (16% memory)
3. **Mixed Precision**: float16 instead of float32 (50% memory)
4. **Gradient Accumulation**: Simulate large batches with small ones
5. **Data Generators**: Load images on-the-fly instead of all at once

> **Key Insight**: Memory constraints often dictate batch size, not optimization considerations. Start with a small batch size and gradually increase until you hit memory limits.

## 9. Lab: The Preprocessor

Now it's your turn to apply what you've learned!

### Task

Create a complete image preprocessing function that:
1. Takes a folder path and target size as input
2. Finds all image files (.jpg, .png) in the folder
3. Processes each image through the full pipeline
4. Returns a batch array ready for model training
5. Reports statistics about the batch

In [ ]:
def create_image_batch(folder_path: str, target_size=(224, 224)):
    """
    Load and preprocess all images in a folder into a batch.
    
    Args:
        folder_path: Path to folder containing images
        target_size: Tuple of (width, height) for resizing
    
    Returns:
        batch: NumPy array of shape (N, H, W, 3) with normalized pixel values
    """
    # TODO: Implement this function
    # Hint: Follow the pipeline we learned:
    # 1. Find all .jpg and .png files
    # 2. For each file:
    #    - Load with cv2.imread()
    #    - Convert BGR → RGB
    #    - Resize to target_size
    #    - Normalize to 0-1
    #    - Append to list
    # 3. Stack list into array with np.stack()
    
    folder = Path(folder_path)
    image_files = sorted(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
    
    if not image_files:
        raise ValueError(f"No images found in {folder_path}")
    
    processed = []
    
    for img_path in image_files:
        # Load
        img = cv2.imread(str(img_path))
        
        # Convert BGR → RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
        
        # Normalize
        img = img.astype(np.float32) / 255.0
        
        processed.append(img)
    
    # Stack into batch
    batch = np.stack(processed, axis=0)
    
    return batch


# Test your function
batch = create_image_batch("data/images", target_size=(224, 224))

print("✓ Batch created successfully!")
print(f"\nBatch statistics:")
print(f"  Shape:      {batch.shape}")
print(f"  Data type:  {batch.dtype}")
print(f"  Value range: {batch.min():.3f} to {batch.max():.3f}")
print(f"  Mean:       {batch.mean():.3f}")
print(f"  Std dev:    {batch.std():.3f}")
print(f"  Memory:     {batch.nbytes / (1024*1024):.2f} MB")

## Summary & Key Takeaways

### What We Learned

1. **Images are Arrays**: Every image is a 3D NumPy array with shape `(Height, Width, Channels)`

2. **Pixels are Numbers**: Color images store three values per pixel (R, G, B) in range 0-255

3. **BGR vs RGB**: OpenCV uses BGR format; always convert to RGB with `cv2.cvtColor()`

4. **Resizing is Essential**: Neural networks need fixed-size inputs; 224×224 is the standard

5. **Normalization Improves Training**: Divide by 255.0 to scale pixel values to 0-1 range

6. **Batching for Efficiency**: Stack multiple images into 4D array `(N, H, W, C)` for GPU processing

7. **Memory Management**: Batch size is constrained by GPU memory; calculate before loading

### The Complete Pipeline

```python
# Load image
img = cv2.imread('photo.jpg')

# Convert BGR → RGB
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize to standard size
img = cv2.resize(img, (224, 224))

# Normalize to 0-1
img = img.astype(np.float32) / 255.0

# Result: (224, 224, 3) array ready for model input
```

### Why This Matters

This preprocessing pipeline is the **foundation of computer vision**:
- Image classification ("What's in this image?")
- Object detection ("Where are the objects?")
- Semantic segmentation ("Which pixels belong to which object?")
- Face recognition, medical imaging, autonomous vehicles, and more

Every computer vision model — from simple CNNs to state-of-the-art transformers — expects images in this processed format.

## Try It Yourself

### Exercise 1: Experiment with Different Sizes

Modify the target size and observe memory changes:

In [ ]:
# Try different sizes
sizes = [(96, 96), (224, 224), (512, 512)]

for size in sizes:
    batch = create_image_batch("data/images", target_size=size)
    mb = batch.nbytes / (1024 * 1024)
    print(f"Size {size[0]}×{size[1]}: {batch.shape} → {mb:.2f} MB")

### Exercise 2: Channel Statistics

Compute mean and standard deviation for each color channel separately:

In [ ]:
batch = create_image_batch("data/images", target_size=(224, 224))

# TODO: Calculate per-channel statistics
# Hint: Use batch[:, :, :, 0] for red channel
#       Use batch[:, :, :, 1] for green channel
#       Use batch[:, :, :, 2] for blue channel

for channel_idx, channel_name in enumerate(['Red', 'Green', 'Blue']):
    channel_data = batch[:, :, :, channel_idx]
    print(f"{channel_name:6s}: mean={channel_data.mean():.3f}, std={channel_data.std():.3f}")

### Exercise 3: Advanced Normalization

Implement ImageNet-style normalization (subtract mean, divide by std per channel):

In [ ]:
# ImageNet statistics (precomputed from millions of images)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_imagenet(batch: np.ndarray) -> np.ndarray:
    """
    Apply ImageNet normalization: (x - mean) / std per channel.
    
    Args:
        batch: Array of shape (N, H, W, 3) with values in [0, 1]
    
    Returns:
        Normalized array with ImageNet statistics
    """
    # TODO: Implement ImageNet normalization
    # Hint: Use broadcasting to subtract mean and divide by std
    # normalized = (batch - mean) / std
    
    normalized = (batch - IMAGENET_MEAN) / IMAGENET_STD
    return normalized


# Test
batch = create_image_batch("data/images", target_size=(224, 224))
batch_imagenet = normalize_imagenet(batch)

print("ImageNet Normalized Batch:")
print(f"  Shape:       {batch_imagenet.shape}")
print(f"  Value range: {batch_imagenet.min():.3f} to {batch_imagenet.max():.3f}")
print(f"  Mean:        {batch_imagenet.mean():.3f}")
print(f"  Std:         {batch_imagenet.std():.3f}")

print(f"\nPer-channel statistics after ImageNet normalization:")
for i, color in enumerate(['Red', 'Green', 'Blue']):
    channel = batch_imagenet[:, :, :, i]
    print(f"  {color:6s}: mean={channel.mean():.3f}, std={channel.std():.3f}")

### Exercise 4: Memory Budget Calculator

Create a function to calculate the maximum batch size given GPU memory:

In [ ]:
def calculate_max_batch_size(gpu_memory_gb: float, 
                             image_size: tuple = (224, 224),
                             memory_safety_factor: float = 0.5) -> int:
    """
    Calculate maximum batch size for given GPU memory.
    
    Args:
        gpu_memory_gb: Available GPU memory in gigabytes
        image_size: (width, height) of images
        memory_safety_factor: Fraction of memory to use (0.5 = 50%)
    
    Returns:
        Maximum batch size
    """
    # TODO: Implement batch size calculator
    # Memory per image = width × height × 3 channels × 4 bytes (float32)
    # Available memory = gpu_memory_gb × safety_factor × (1024^3) bytes
    # Max batch size = available_memory / memory_per_image
    
    H, W = image_size
    bytes_per_image = H * W * 3 * 4  # float32 = 4 bytes
    available_bytes = gpu_memory_gb * memory_safety_factor * (1024 ** 3)
    max_batch = int(available_bytes / bytes_per_image)
    
    return max_batch


# Test with different GPU configurations
print("Maximum batch sizes for 224×224 images:\n")
print(f"{'GPU Memory':>15} | {'50% Usage':>12} | {'80% Usage':>12}")
print("-" * 45)

for gpu_gb in [4, 8, 12, 24, 40, 80]:
    batch_50 = calculate_max_batch_size(gpu_gb, memory_safety_factor=0.5)
    batch_80 = calculate_max_batch_size(gpu_gb, memory_safety_factor=0.8)
    print(f"{gpu_gb:>15} GB | {batch_50:>12,} | {batch_80:>12,}")

## Next Steps

After mastering image preprocessing, you can explore:

1. **Data Augmentation**: Random transformations to increase dataset diversity
   - Rotation, flipping, cropping
   - Color jittering, brightness adjustment
   - Mixup, CutMix for regularization

2. **Efficient Data Loading**: Handle large datasets
   - PyTorch DataLoader with multiprocessing
   - TensorFlow tf.data pipelines
   - Image generators for RAM management

3. **Transfer Learning**: Use pretrained models
   - Load ResNet, VGG, or EfficientNet
   - Fine-tune on your dataset
   - Apply to new domains

4. **Real-World Applications**: Build complete systems
   - Image classification (CIFAR-10, ImageNet)
   - Object detection (YOLO, Faster R-CNN)
   - Semantic segmentation (U-Net, DeepLab)

### Additional Resources

- [OpenCV Python Tutorial](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [NumPy for Image Processing](https://numpy.org/doc/stable/user/howtos_io.html)
- [PyTorch Vision Transforms](https://pytorch.org/vision/stable/transforms.html)
- [TensorFlow Image Preprocessing](https://www.tensorflow.org/tutorials/load_data/images)
- [ImageNet Dataset](https://www.image-net.org/)

---

**Congratulations!** You now understand how computers see images and how to prepare them for machine learning models. This is a fundamental skill for any computer vision engineer. 🎉